![Multi_GPU_Slide](img/HPCP_MultiGPU/Folie1.PNG) 

![Multi_GPU_Slide](img/HPCP_MultiGPU/Folie2.PNG) 

![Multi_GPU_Slide](img/HPCP_MultiGPU/Folie3.PNG) 

![Multi_GPU_Slide](img/HPCP_MultiGPU/Folie4.PNG)

In [1]:
!mkdir scripts

In [2]:
%%writefile scripts/run_script.sh
#!/bin/bash

module load anaconda
cd ${HOME}/HS26/04_Advanced/scripts
export PYTHONPATH=${HOME}/HS26/04_Advanced/scripts:$PYTHONPATH
conda activate summer-school-hpc-2025
mpirun python ${SCRIPT_NAME}.py

Writing scripts/run_script.sh


In [3]:
!chmod +x scripts/run_script.sh

we will use this submission script for all our tests like this:

```bash
cd HS26/04_Advanced/scripts
SCRIPT_NAME=single_cp sbatch --cluster=gmerlin7 --partition=a100-hourly --reservation=psicourse01 --gpus=2 --cpus-per-task=2 --output=log.out run_script.sh
```

Let us discuss the following simple example.

In [4]:
%%writefile scripts/single_cp.py
import cupy as cp
import numpy as np

print(f"Hello from single_job. I see {cp.cuda.runtime.getDeviceCount()} devices.")

A = cp.random.random((1024,1024), dtype=cp.float32)
A = cp.random.random((1024,1024), dtype=cp.float32)
A = cp.random.random((1024,1024), dtype=cp.float32)
A = cp.random.random((1024,1024), dtype=cp.float32)

with cp.cuda.Device(0):   # select GPU 0
    A = cp.random.random((1024,1024), dtype=cp.float32)
    B = cp.random.random((1024,1024), dtype=cp.float32)
    res1_gpu = A @ B 

with cp.cuda.Device(1):   # select GPU 1
    C = cp.random.random((1024,1024), dtype=cp.float32)
    D = cp.random.random((1024,1024), dtype=cp.float32)
    res2_gpu = C @ D 

res1_host = cp.asnumpy(res1_gpu) 
res2_host = cp.asnumpy(res2_gpu) 
print(f"Done. res1 preview: {res1_host[0:5]} \nand res2 preview: {res2_host[0:5]}")

Writing scripts/single_cp.py


In [19]:
!cat scripts/log.out

Hello from single_job. I see 2 devices.
Done. res1 preview: [[243.97412 248.05185 249.9029  ... 261.0253  261.1789  255.34895]
 [253.66034 248.2228  252.39926 ... 262.94287 266.6966  250.7923 ]
 [261.63257 256.16394 263.0752  ... 269.082   274.162   262.13193]
 [252.28265 245.14882 252.74081 ... 257.71622 262.12726 255.1165 ]
 [246.93062 240.59546 245.90204 ... 253.8186  255.9559  251.0174 ]] and res2 preview: [[254.87265 258.94232 255.34753 ... 266.2976  254.3287  262.9796 ]
 [244.70996 262.54272 254.03656 ... 260.9129  250.0023  259.76602]
 [246.9503  253.97754 251.55046 ... 258.73566 252.97365 255.2243 ]
 [255.05783 262.63297 262.20947 ... 270.6292  256.50345 262.2771 ]
 [249.85574 264.8111  260.9229  ... 256.94852 257.7815  264.13885]]


Let us move some data from GPU 0 to GPU 1. 

In [5]:
%%writefile scripts/p2p_cp.py
import cupy as cp
from cupy.cuda import runtime, Device, Stream

src_dev, dst_dev = 0, 1

#Enable P2P access
with Device(dst_dev):
    runtime.deviceEnablePeerAccess(src_dev)
with Device(src_dev):
    runtime.deviceEnablePeerAccess(dst_dev)

with Device(src_dev):
    a = cp.arange(10_000_000, dtype=cp.float32)

# Allocate destination on GPU 1
with Device(dst_dev):
    b = cp.empty_like(a)
    runtime.memcpyPeerAsync(b.data.ptr, dst_dev, a.data.ptr, src_dev, a.nbytes, Stream.null.ptr)
    Stream.null.synchronize() # wait until done (this is in-efficient, but OK for this example)

b_host = cp.asnumpy(b) 
print(f"Done cpying data from Device 0 to 1. b: {b_host[0:5]}")

Writing scripts/p2p_cp.py


```bash
SCRIPT_NAME=p2p_cp sbatch --cluster=gmerlin7 --partition=a100-hourly --gpus=2 --reservation=psicourse01 --cpus-per-task=2 --output=log.out run_script.sh
```

This is pretty tedious—better to use a library that supports communication patterns.

![Multi_GPU_Slide](img/HPCP_MultiGPU/Folie5.PNG)

![Multi_GPU_Slide](img/HPCP_MultiGPU/Folie6.PNG) 

![Multi_GPU_Slide](img/HPCP_MultiGPU/Folie7.PNG) 

![Multi_GPU_Slide](img/HPCP_MultiGPU/Folie8.PNG) 

#### Task 1 — Implement `allGather` with NCCL

Follow the official documentation:  
- CuPy NCCL docs: https://docs.cupy.dev/en/stable/reference/generated/cupy.cuda.nccl.NcclCommunicator.html  
- NVIDIA NCCL user guide: https://docs.nvidia.com/deeplearning/nccl/user-guide/docs/index.html  

**Steps:**
1. Create send and receive buffers on all GPUs.  
2. Initialize NCCL across all GPUs with `initAll`.  
3. Call the `allGather` operation on each GPU.  
   - *Hint:* Wrap these calls inside `groupStart()` and `groupEnd()`.  
4. Synchronize, then print part of the result to verify correctness.  

In [6]:
%%writefile scripts/nccl_allgather.py
import cupy as cp
from cupy.cuda import nccl, Device, Stream
from cupy import cuda

devs = [0, 1]
n = 1024
dtype = cp.float32
dtype_nccl = nccl.NCCL_FLOAT32

# Create per-device buffers
bufs = []
outs = []
for d in devs:
    with Device(d):
        bufs.append(cp.arange(n, dtype=dtype))
        outs.append(cp.empty((2,n), dtype=dtype))

# Init a NCCL comm across these devices (ranks are 0..len(devs)-1)
comms = nccl.NcclCommunicator.initAll(len(devs))
print(comms)

# Example: all-gather across the two GPUs (each gets both buffers)
cuda.nccl.groupStart()
for i in range(2):
    with Device(i):
        print(f"Start allGatrher on: {comms[i]}")
        comms[i].allGather(bufs[i].data.ptr, outs[i].data.ptr, n, dtype_nccl, cuda.Stream.null.ptr)
cuda.nccl.groupEnd()

for i in range(2):
    with Device(i) as device:
        print("sync")
        device.synchronize()

print(f"Done: {outs[i][1][0:5]}")

Writing scripts/nccl_allgather.py


```bash
SCRIPT_NAME=nccl_allgather sbatch --cluster=gmerlin7 --partition=a100-hourly --reservation=psicourse01 --gpus=2 --cpus-per-task=2 --output=log.out run_script.sh
```

In [8]:
!cat scripts/log.out

[<cupy_backends.cuda.libs.nccl.NcclCommunicator object at 0x14f2c3da3830>, <cupy_backends.cuda.libs.nccl.NcclCommunicator object at 0x14f2c3da3d90>]
Start allGatrher on: <cupy_backends.cuda.libs.nccl.NcclCommunicator object at 0x14f2c3da3830>
Start allGatrher on: <cupy_backends.cuda.libs.nccl.NcclCommunicator object at 0x14f2c3da3d90>
sync
sync
Done: [0. 1. 2. 3. 4.]


#### Task 2

Write an NCCL script where each GPU is controlled by its own process.  
You can achieve this by either:
- running a single task with **N cores and N GPUs**, or  
- running **N tasks**, each bound to one GPU.  

**Requirements:**
- Broadcast an array from **rank/GPU 0** to all other ranks.  
- No `groupStart()` needed, but **each process must initialize** its own `NcclCommunicator`.  
- Use `comm_id = nccl.get_unique_id()` to initialize the communicator.  
- Use `cuda.Stream.null.ptr` as the CUDA stream (default stream).  
- Use `n_devices = int(os.environ["SLURM_GPUS_ON_NODE"])` to get the number of GPUs and **not** `cp.cuda.runtime.getDeviceCount()`! Calling cuda.runtime before starting new processes will lead to an error.
- Use `comm.destroy()` to destroy the `NcclCommunicator` within each process at the end

In [9]:
%%writefile scripts/nccl_broadcast.py
import multiprocessing
import os

import cupy as cp
from cupy import cuda
from cupy.cuda import nccl
from cupy import testing

def f(n_devices, comm_id, rank):
    print(f"Rank {rank} started.")
    device = cuda.Device(rank)
    device.use()
    comm = nccl.NcclCommunicator(n_devices, comm_id, rank)
    recv = cp.zeros(1024, dtype='float32')
    comm.broadcast(recv.data.ptr, recv.data.ptr, recv.size, nccl.NCCL_FLOAT, 0, cuda.Stream.null.ptr)
    device.synchronize()
    print(f"Rank {rank} finished: {recv[0:5]}")
    comm.destroy()

if __name__ == '__main__':
    n_devices = int(os.environ["SLURM_GPUS_ON_NODE"])
    print(f"Found {n_devices} GPUs")
    comm_id = nccl.get_unique_id()

    ps = []
    for i in range(1, n_devices):
        p = multiprocessing.Process(target=f, args=(n_devices, comm_id, i))
        p.start()
        ps.append(p)

    device = cuda.Device(0)
    device.use()
    comm = nccl.NcclCommunicator(n_devices, comm_id, 0)
    send = cp.ones(1024, dtype='float32')
    comm.broadcast(send.data.ptr, send.data.ptr, send.size, nccl.NCCL_FLOAT, 0, cuda.Stream.null.ptr)

    for p in ps:
        p.join(10)
    
    print('Rank 0 successfully finished.')
    comm.destroy()

Writing scripts/nccl_broadcast.py


```bash
SCRIPT_NAME=nccl_broadcast sbatch --cluster=gmerlin7 --partition=a100-hourly --reservation=psicourse01 --gpus=4 --cpus-per-task=4 --output=log.out run_script.sh
```

In [24]:
!cat scripts/log.out

Found 4 GPUs
Rank 3 started.
Rank 2 started.
Rank 1 started.
Rank 1 finished: [1. 1. 1. 1. 1.]
Rank 2 finished: [1. 1. 1. 1. 1.]
Rank 3 finished: [1. 1. 1. 1. 1.]
Rank 0 successfully finished.


![Multi_GPU_Slide](img/HPCP_MultiGPU/Folie9.PNG)

In [10]:
%%writefile scripts/hello_mpi.py
from mpi4py import MPI
import cupy as cp
from cupy import cuda
import socket

comm = MPI.COMM_WORLD
size = comm.Get_size()
rank = comm.Get_rank()

num_gpus = cuda.runtime.getDeviceCount()
print(f"Hello from rank {rank} on {socket.gethostname()}, I see {num_gpus} GPU(s).")
cp.cuda.Device(rank % num_gpus).use()  # Each task gets an isolated GPU

Writing scripts/hello_mpi.py


```bash
SCRIPT_NAME=hello_mpi sbatch --cluster=gmerlin7 --partition=a100-hourly --reservation=psicourse01 --ntasks=4 --gpus-per-task=1 --output=log.out run_script.sh
```

In [1]:
!cat scripts/log.out

Hello from rank 3 on gpu105, I see 4 GPU(s).
Hello from rank 2 on gpu105, I see 4 GPU(s).
Hello from rank 0 on gpu105, I see 4 GPU(s).
Hello from rank 1 on gpu105, I see 4 GPU(s).


#### Task 3
Understand how Slurm parameters affect resource allocation and rank→GPU mapping.
Launch short jobs with different combinations of:
* --ntasks
* --gpus-per-task
* --ntasks-per-node=4
* --ngpus

In [ ]:
#ToDo: Play around

#### Task 4
Use two MPI ranks (0 and 1), one GPU per rank. Create a CuPy buffer on rank 0 and send it to rank 1.

* Initialize MPI and map each rank to a GPU.
* On rank 0: create a CuPy array (e.g., cp.arange(...)) on its GPU.
* Send that device buffer to rank 1 using comm.Send (CUDA-aware path: `comm.Send([data, MPI.FLOAT], ...)`).
* On rank 1: Recv into a CuPy buffer on its GPU and verify contents (cp.allclose, print a small slice).

In [8]:
%%writefile scripts/send_mpi.py
from mpi4py import MPI
import cupy as cp
import numpy as np
import os

comm = MPI.COMM_WORLD
size = comm.Get_size()
rank = comm.Get_rank()
num_gpus = cp.cuda.runtime.getDeviceCount()
print(f"Hello from {rank}. I see {cp.cuda.runtime.getDeviceCount()} devices.")
print(f"Rank {rank}: {os.environ['CUDA_VISIBLE_DEVICES']}")

if rank == 0:
    cp.cuda.Device(0).use()
    data = cp.arange(10, dtype=np.float32)
    comm.Send([data, MPI.FLOAT], dest=1, tag=13)
    print ("0 sent:", data)
elif rank == 1:
    cp.cuda.Device(1).use()
    recv_data = cp.empty(10, dtype=np.float32)
    comm.Recv([recv_data, MPI.FLOAT], source=0, tag=13)
    print ("1 recv:",recv_data)

Overwriting scripts/send_mpi.py


In order to run this we need a new conda env which has special built mpi and cuda libs to run on the Merlin7 cluster.

In [ ]:
%%writefile scripts/run_script_v2.sh
#!/bin/bash

module purge
module use Alps_A100
module load anaconda hwloc/2.12.0 gcc/14.3.0 mpich/5.0.1

cd ${HOME}/HS26/04_Advanced/scripts
export PYTHONPATH=${HOME}/HS26/04_Advanced/scripts:$PYTHONPATH
conda activate summer-school-hpc-2026
mpirun python ${SCRIPT_NAME}.py

```bash
SCRIPT_NAME=send_mpi sbatch --cluster=gmerlin7 --partition=a100-hourly --reservation=psicourse01 --ntasks=2 --gpus-per-task=1 --output=log.out run_script_v2.sh
```

In [5]:
!cat scripts/log.out

Hello from 0. I see 2 devices.
Rank 0: 0,1
Hello from 1. I see 2 devices.
Rank 1: 0,1
[gpu104:14641:0:14641] Caught signal 11 (Segmentation fault: invalid permissions for mapped object at address 0x1553df200000)
==== backtrace (tid:  14641) ====
 0  /opt/psi/Programming/anaconda/2024.08/conda/envs/summer-school-hpc-2025/lib/python3.12/site-packages/mpi4py/../../.././libucs.so.0(ucs_handle_error+0x2fd) [0x15542b1fa78d]
 1  /opt/psi/Programming/anaconda/2024.08/conda/envs/summer-school-hpc-2025/lib/python3.12/site-packages/mpi4py/../../.././libucs.so.0(+0x2f981) [0x15542b1fa981]
 2  /opt/psi/Programming/anaconda/2024.08/conda/envs/summer-school-hpc-2025/lib/python3.12/site-packages/mpi4py/../../.././libucs.so.0(+0x2fb4a) [0x15542b1fab4a]
 3  /lib64/libc.so.6(+0x57980) [0x15542c2af980]
 4  /lib64/libc.so.6(+0x17e019) [0x15542c3d6019]
 5  /opt/psi/Programming/anaconda/2024.08/conda/envs/summer-school-hpc-2025/lib/python3.12/site-packages/mpi4py/../../.././libopen-pal.so.80(opal_convertor_p

![Multi_GPU_Slide](img/HPCP_MultiGPU/Folie10.PNG)